In [1]:
!apt-get update
!apt-get install -y xvfb ffmpeg
!apt-get install xvfb-run -s
!apt-get install -y xvfb x11-utils
!pip install 'imageio==2.4.0'
!pip install pyvirtualdisplay
!pip install tf-agents
!pip install keras-rl2
!pip install gym
!pip install h5py
!pip install stable-baselines3[extra]

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [110 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [119 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [108 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [802 kB]
Hit:8 https://ppa.launchpadcontent.net/c2d4u.team/c2d4u4.0+/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,230 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [848 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [969 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [1,087 kB]
Get:13 http://archive.ubuntu

In [2]:
import os
import io
import time
import base64
import imageio
import IPython
import shutil
import numpy as np
import matplotlib as mp
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
import gym
import random
from gym import Env
from gym.spaces import Discrete, Box
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnRewardThreshold
import tensorflow as tf
from tensorflow import keras
import tensorflow_hub as hub
import tensorflow_datasets as tfds
from tf_agents.agents.dqn import dqn_agent
from tf_agents.policies import random_tf_policy
from tf_agents.networks import sequential
from tf_agents.environments import suite_gym
from tf_agents.environments import tf_py_environment
from tf_agents.eval import metric_utils
from tf_agents.metrics import tf_metric
from rl.agents.dqn import DQNAgent
from rl.policy import EpsGreedyQPolicy, BoltzmannQPolicy
from rl.memory import SequentialMemory
from google.colab import files
import pyvirtualdisplay
# file_load = files.upload()
# for f in file_load.keys():
#   print('Uploaded file "{Name}" with length {length}.bytes'.format(
#       name =f, length = len(file_load[f])))
print(tf.__version__)

class ctrl_env(Env):
  def __init__(self):
    self.observation_space = Box(low=np.array([0]), high =np.array([100]), shape=(1,))
    self.action_space = Discrete(3)
    self.state = 50+random.randint(-5,5)
    self.episode_length = 50

  def reset(self):
    self.state = 50+random.randint(-5,5)
    self.episode_length = 50
    return self.state

  def step(self,action):
    self.state+= action-1
    self.episode_length-= 1
    if self.state>=47 and self.state<=53:
      reward = 1
    else:
      reward=-1
    if self.episode_length<=0:
      done=True
    else:
      done= False
    info={}
    return self.state, reward, done, info

  def render():
    pass

2.12.0


In [3]:
env=ctrl_env()
episodes =20
for episode in range(1, episodes+1):
  state=env.reset()
  done=False
  tot_reward=0
  while not done:
    action=env.action_space.sample()
    state,reward,done,info = env.step(action)
    tot_reward+=reward
  print('Episode: {}, Total Reward: {}'.format(episode, tot_reward))

Episode: 1, Total Reward: 46
Episode: 2, Total Reward: -24
Episode: 3, Total Reward: 2
Episode: 4, Total Reward: -42
Episode: 5, Total Reward: -12
Episode: 6, Total Reward: 48
Episode: 7, Total Reward: 40
Episode: 8, Total Reward: 46
Episode: 9, Total Reward: 20
Episode: 10, Total Reward: 38
Episode: 11, Total Reward: 48
Episode: 12, Total Reward: 20
Episode: 13, Total Reward: 50
Episode: 14, Total Reward: 24
Episode: 15, Total Reward: 4
Episode: 16, Total Reward: 50
Episode: 17, Total Reward: -38
Episode: 18, Total Reward: 2
Episode: 19, Total Reward: -26
Episode: 20, Total Reward: 46


/usr/local/lib/python3.10/dist-packages/gym/spaces/box.py:84: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


In [4]:
log_path=os.path.join('Training','Logs')
# add_policy=[dict(pi=[256,256,128,64],vf=[256,256,128,64])]
agent=PPO('MlpPolicy',env,verbose=1,tensorboard_log=log_path)
agent.learn(total_timesteps=10000)
# !tensorboard --logdir=log_path

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


Logging to Training/Logs/PPO_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -0.7     |
| time/              |          |
|    fps             | 548      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 50           |
|    ep_rew_mean          | -1.14        |
| time/                   |              |
|    fps                  | 391          |
|    iterations           | 2            |
|    time_elapsed         | 10           |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0024793027 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.09        |
|    explained_variance   |

In [5]:
# agent_path=os.path.join('Training', 'Models','PPO')
# agent.save(agent_path)
# del agent
#agent=PPO.load(log_path,env=env)

In [6]:
evaluate_policy(agent,env,n_eval_episodes=10,render=False)

/usr/local/lib/python3.10/dist-packages/stable_baselines3/common/evaluation.py:67: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


(50.0, 0.0)